In [ ]:
# ================================================================
# Direct file read from Colab
# Figure texts translated to English
# ================================================================

# If needed, uncomment:
# !pip install -q pandas numpy matplotlib seaborn plotly scipy scikit-learn

import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns
from scipy.stats import chi2_contingency
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer

warnings.filterwarnings("ignore")

# ---------------------------------------------------------------
# FILE PATH IN COLAB
# ---------------------------------------------------------------
file_path = "Table1"


def load_csv() -> pd.DataFrame:
    df = pd.read_csv(file_path, low_memory=False)
    print(f"File loaded successfully: {file_path}")
    print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")
    return df


# ---------------------------------------------------------------
# FIGURE 1 — Chronic diseases vs injectable drug use
# ---------------------------------------------------------------
def figure_1_chronic_vs_injectables(df: pd.DataFrame) -> None:
    text_cols = [c for c in df.columns if df[c].dtype == object]
    df_text = df[text_cols].fillna("")

    regex_inject = r"\b(injet[aá]ve[li]s?|udvi)\b"
    cond_inject_text = df_text.apply(
        lambda x: x.astype(str).str.contains(regex_inject, case=False, regex=True)
    ).any(axis=1)

    drug_cols = [c for c in df.columns if "droga" in c.lower() or "injet" in c.lower() or "comportamento" in c.lower()]
    cond_inject_struct = pd.Series(False, index=df.index)
    for c in drug_cols:
        if "injet" in c.lower() and "choice=" in c.lower():
            is_checked = df[c].astype(str).str.lower().str.strip().isin(
                ["checked", "1", "1.0", "sim", "verdadeiro", "true", "yes"]
            )
            cond_inject_struct = cond_inject_struct | is_checked

    df = df.copy()
    df["Injectable_Use"] = (cond_inject_text | cond_inject_struct).astype(int)

    regex_htn = r"\b(hipertens[aã]o|has|press[aã]o alta)\b"
    regex_dm = r"\b(diabetes|dm2|dm1|dm)\b"

    cond_htn_text = df_text.apply(
        lambda x: x.astype(str).str.contains(regex_htn, case=False, regex=True)
    ).any(axis=1)
    cond_dm_text = df_text.apply(
        lambda x: x.astype(str).str.contains(regex_dm, case=False, regex=True)
    ).any(axis=1)

    chronic_cols = [c for c in df.columns if "doenças crônicas" in c.lower() and "choice=" in c.lower()]
    cond_htn_struct = pd.Series(False, index=df.index)
    cond_dm_struct = pd.Series(False, index=df.index)

    for c in chronic_cols:
        is_checked = df[c].astype(str).str.lower().str.strip().isin(
            ["checked", "1", "1.0", "sim", "verdadeiro", "true", "yes"]
        )
        if "hipertensão" in c.lower() or "has" in c.lower():
            cond_htn_struct = cond_htn_struct | is_checked
        if "diabetes" in c.lower() or "dm" in c.lower():
            cond_dm_struct = cond_dm_struct | is_checked

    df["Has_Hypertension"] = (cond_htn_text | cond_htn_struct).astype(int)
    df["Has_Diabetes"] = (cond_dm_text | cond_dm_struct).astype(int)

    df_patients = df.groupby("Record ID")[["Injectable_Use", "Has_Hypertension", "Has_Diabetes"]].max().reset_index()
    df_patients["Injectable_History"] = np.where(df_patients["Injectable_Use"] == 1, "Yes", "No")

    def calculate_pvalue(disease_column: str):
        contingency = pd.crosstab(df_patients["Injectable_Use"], df_patients[disease_column])
        if contingency.shape == (2, 2):
            _, p, _, _ = chi2_contingency(contingency, correction=False)
            return round(p, 4)
        return "N/A"

    p_htn = calculate_pvalue("Has_Hypertension")
    p_dm = calculate_pvalue("Has_Diabetes")

    results = []
    for group in ["Yes", "No"]:
        df_group = df_patients[df_patients["Injectable_History"] == group]
        total_patients = len(df_group)

        if total_patients > 0:
            htn_abs = df_group["Has_Hypertension"].sum()
            dm_abs = df_group["Has_Diabetes"].sum()

            results.append(
                {
                    "Comorbidity": "Hypertension",
                    "Injectable_History": group,
                    "Total_Unique_Patients": total_patients,
                    "Confirmed_Cases": htn_abs,
                    "Prevalence_%": round((htn_abs / total_patients) * 100, 2),
                    "P_Value": p_htn,
                }
            )

            results.append(
                {
                    "Comorbidity": "Diabetes",
                    "Injectable_History": group,
                    "Total_Unique_Patients": total_patients,
                    "Confirmed_Cases": dm_abs,
                    "Prevalence_%": round((dm_abs / total_patients) * 100, 2),
                    "P_Value": p_dm,
                }
            )

    df_table = pd.DataFrame(results)

    print("\n" + "=" * 95)
    print(f"CHART TABLE: CHRONIC DISEASES vs INJECTABLE USE (N={len(df_patients)} UNIQUE PATIENTS)")
    print("=" * 95)
    print(df_table.to_string(index=False))
    print("=" * 95)

    output_file = "Chronic_Disease_Prevalence_vs_Injectables_UNIQUE_PATIENTS_EN.csv"
    df_table.to_csv(output_file, index=False, encoding="utf-8-sig")
    print(f"\n[SUCCESS] The table used to generate the chart was saved as '{output_file}'.\n")

    sns.set_theme(style="whitegrid")
    plt.figure(figsize=(9, 6))
    colors = ["#d05c53", "#95a5a6"]

    ax = sns.barplot(
        data=df_table,
        x="Comorbidity",
        y="Prevalence_%",
        hue="Injectable_History",
        palette=colors,
        edgecolor="white",
        linewidth=1.5,
    )

    total_unique = len(df_patients)
    injectable_unique = len(df_patients[df_patients["Injectable_Use"] == 1])

    plt.title(
        f"Prevalence of Chronic Diseases: Injectable Drug Users vs General Population\n"
        f"(Base: {total_unique} Unique Patients | {injectable_unique} IDUs)",
        fontsize=14,
        pad=20,
        fontweight="bold",
        color="#2c3e50",
    )
    plt.ylabel("Prevalence Within Group (%)", fontsize=12, fontweight="bold", color="#34495e")
    plt.xlabel("")
    plt.ylim(0, max(df_table["Prevalence_%"]) * 1.25)

    for container in ax.containers:
        ax.bar_label(container, fmt="%.1f%%", padding=4, fontweight="bold", color="#2c3e50", fontsize=11)

    sns.despine(left=True)
    plt.legend(
        title="History of Injectable Use",
        title_fontsize="11",
        fontsize="10",
        loc="upper right",
        frameon=True,
        shadow=False,
    )
    plt.tight_layout()
    plt.show()


# ---------------------------------------------------------------
# FIGURE 2A — CIAP-2 cascade to referrals
# FIGURE 2B — consultation time distortion
# ---------------------------------------------------------------
def figure_2_ciap_and_time(df: pd.DataFrame) -> None:
    cols_ciap = [c for c in df.columns if "ciap" in c.lower()]
    cols_espec = [c for c in df.columns if "Para qual especialidade" in c and "choice=" in c]
    cols_time = [c for c in df.columns if "tempo" in c.lower() or "duração" in c.lower() or "duracao" in c.lower()]

    df_ev = df[df["Repeat Instrument"].notna()].copy()

    # PART A — CIAP -> referrals
    if cols_ciap and cols_espec:
        col_ciap = cols_ciap[0]

        df_ciap = df_ev.dropna(subset=[col_ciap]).copy()
        df_ciap = df_ciap[df_ciap[col_ciap].astype(str).str.strip() != ""]
        df_ciap = df_ciap[df_ciap[col_ciap].astype(str).str.lower() != "nan"]

        flow_records = []

        for _, row in df_ciap.iterrows():
            selected_specialties = [c for c in cols_espec if row[c] == "Checked"]
            current_ciap = str(row[col_ciap])

            if len(selected_specialties) == 0:
                destination = "No Referral (Resolved)"
                referred = 0
            elif len(selected_specialties) == 1:
                destination = selected_specialties[0].split("choice=")[1].replace(")", "").strip()
                referred = 1
            else:
                destination = "Multiple Referrals"
                referred = 1

            flow_records.append(
                {
                    "CIAP": current_ciap,
                    "Destination": destination,
                    "Referred": referred,
                }
            )

        df_flow = pd.DataFrame(flow_records)

        top_ciaps = df_flow["CIAP"].value_counts().nlargest(10).index
        top_specialties = df_flow["Destination"].value_counts().nlargest(10).index

        df_flow["CIAP_Plot"] = np.where(df_flow["CIAP"].isin(top_ciaps), df_flow["CIAP"], "Other CIAPs (Grouped)")
        df_flow["Destination_Plot"] = np.where(
            df_flow["Destination"].isin(top_specialties), df_flow["Destination"], "Other Specialties"
        )
        df_flow["CIAP_Plot"] = df_flow["CIAP_Plot"].apply(lambda x: x[:40] + "..." if len(x) > 40 else x)

        fig_cascade = px.parallel_categories(
            df_flow,
            dimensions=["CIAP_Plot", "Destination_Plot"],
            color="Referred",
            color_continuous_scale=[(0.0, "#2ecc71"), (1.0, "#e74c3c")],
            labels={
                "CIAP_Plot": "Initial Diagnosis (CIAP-2)",
                "Destination_Plot": "Post-Consultation Destination",
            },
            title=f"Resolution and Referrals by CIAP-2 (N={len(df_flow)} consultations)",
        )

        fig_cascade.update_layout(
            height=800,
            font=dict(size=11, family="Arial"),
            coloraxis_showscale=False,
            margin=dict(l=300, r=200, t=100, b=50),
        )
        fig_cascade.show()

        df_flow_table = (
            df_flow.groupby(["CIAP_Plot", "Destination_Plot", "Referred"])
            .size()
            .reset_index(name="Consultation_Count")
        )

        df_flow_table["Resolution_Status"] = df_flow_table["Referred"].map(
            {0: "Resolved in Primary Care", 1: "Referred"}
        )
        df_flow_table = df_flow_table.sort_values(
            by=["Referred", "Consultation_Count"], ascending=[True, False]
        )
        df_flow_table = df_flow_table[
            ["CIAP_Plot", "Resolution_Status", "Destination_Plot", "Consultation_Count"]
        ].reset_index(drop=True)

        print("\n" + "=" * 95)
        print("TABLE 1: RESOLUTION AND REFERRAL FLOW BY CIAP-2")
        print("=" * 95)
        print(df_flow_table.to_string(index=False))

        output_flow = "CIAP_Resolution_Referral_Flow_Table_EN.csv"
        df_flow_table.to_csv(output_flow, index=False, encoding="utf-8-sig")
        print(f"-> Saved as: {output_flow}\n")

    else:
        print("CIAP or specialty columns were not found to generate Figure 2A.")

    # PART B — consultation time distortion
    if cols_time:
        col_time = cols_time[0]

        def extract_minutes(value):
            try:
                val_str = str(value)
                numbers = re.findall(r"\d+", val_str)
                return float(numbers[0]) if numbers else np.nan
            except Exception:
                return np.nan

        df_ev["Time_Minutes"] = df_ev[col_time].apply(extract_minutes)
        df_valid_time = df_ev[(df_ev["Time_Minutes"] >= 1) & (df_ev["Time_Minutes"] <= 300)]

        if not df_valid_time.empty:
            plt.figure(figsize=(12, 6))
            sns.set_theme(style="whitegrid")

            ax = sns.violinplot(x=df_valid_time["Time_Minutes"], color="#3498db", inner="box")

            median_val = df_valid_time["Time_Minutes"].median()
            mean_val = df_valid_time["Time_Minutes"].mean()

            plt.axvline(median_val, color="red", linestyle="--", label=f"Median ({median_val:.1f} min)")
            plt.axvline(mean_val, color="green", linestyle=":", label=f"Mean ({mean_val:.1f} min)")

            plt.title("Consultation Time Distortion (Teleconsultations)", fontsize=15, pad=15, fontweight="bold")
            plt.xlabel(f'Consultation Duration (minutes) - Column: "{col_time}"', fontsize=12)
            plt.legend()
            plt.tight_layout()
            plt.show()

            df_stats = df_valid_time["Time_Minutes"].describe().round(1).reset_index()
            df_stats.columns = ["Metric", "Value (Minutes)"]

            metric_map = {
                "count": "Total Consultations Analyzed",
                "mean": "Mean Time",
                "std": "Standard Deviation (Variation/Distortion)",
                "min": "Minimum Time",
                "25%": "25th Percentile (Q1)",
                "50%": "Median (50th Percentile)",
                "75%": "75th Percentile (Q3)",
                "max": "Absolute Maximum Time",
            }
            df_stats["Metric"] = df_stats["Metric"].map(metric_map)

            print("\n" + "=" * 80)
            print("TABLE 2: STATISTICAL SUMMARY OF CONSULTATION TIME")
            print("=" * 80)
            print(df_stats.to_string(index=False))

            print("\n* MANAGEMENT INTERPRETATION:")
            print("The distance between the mean and the median in the chart above, together with the length")
            print("of the violin 'neck', indicates how many consultations fall outside the normal curve")
            print("(distortion). A few very long cases pull the mean upward, while most consultations")
            print("remain concentrated in the wider central body of the chart.")

            output_time = "Consultation_Time_Statistics_Table_EN.csv"
            df_stats.to_csv(output_time, index=False, encoding="utf-8-sig")
            print(f"\n-> Saved as: {output_time}\n")

        else:
            print("The time column was found, but the values could not be converted into numeric minutes.")
    else:
        print("No column containing 'tempo' or 'duração' was found to generate Figure 2B.")


# ---------------------------------------------------------------
# FIGURE 3 — CIAP flow to Ophthalmology vs Endocrinology
# ---------------------------------------------------------------
def figure_3_ophthalmo_vs_endocrino(df: pd.DataFrame) -> None:
    df_ev = df[df["Repeat Instrument"].notna()].copy()

    cols_ciap = [c for c in df.columns if "ciap" in c.lower()]
    if not cols_ciap:
        raise ValueError("No CIAP column was found.")
    col_ciap = cols_ciap[0]

    cols_espec = [c for c in df.columns if "Para qual especialidade" in c and "choice=" in c]
    cols_oftalmo = [c for c in cols_espec if "oftalmo" in c.lower()]
    cols_endocrino = [c for c in cols_espec if "endocrin" in c.lower()]

    df_ev = df_ev.dropna(subset=[col_ciap])
    df_ev = df_ev[df_ev[col_ciap].astype(str).str.strip() != ""]
    df_ev = df_ev[df_ev[col_ciap].astype(str).str.lower() != "nan"]

    flow_records = []

    for _, row in df_ev.iterrows():
        current_ciap = str(row[col_ciap])

        referred_oft = any(row[c] == "Checked" for c in cols_oftalmo)
        if referred_oft:
            flow_records.append(
                {"CIAP": current_ciap, "Specialty": "Ophthalmology", "Flow_Color": 0}
            )

        referred_endo = any(row[c] == "Checked" for c in cols_endocrino)
        if referred_endo:
            flow_records.append(
                {"CIAP": current_ciap, "Specialty": "Endocrinology", "Flow_Color": 1}
            )

    df_flow = pd.DataFrame(flow_records)

    if not df_flow.empty:
        top_ciaps = df_flow["CIAP"].value_counts().nlargest(15).index
        df_flow["CIAP_Plot"] = np.where(df_flow["CIAP"].isin(top_ciaps), df_flow["CIAP"], "Other CIAPs (Grouped)")
        df_flow["CIAP_Plot"] = df_flow["CIAP_Plot"].apply(lambda x: x[:45] + "..." if len(x) > 45 else x)

        fig = px.parallel_categories(
            df_flow,
            dimensions=["CIAP_Plot", "Specialty"],
            color="Flow_Color",
            color_continuous_scale=[(0.0, "#3498db"), (1.0, "#e67e22")],
            labels={
                "CIAP_Plot": "Base Diagnosis (CIAP-2)",
                "Specialty": "Destination Specialty",
            },
            title=f"Referral Flow: Ophthalmology vs Endocrinology (N={len(df_flow)} flows)",
        )

        fig.update_layout(
            height=700,
            font=dict(size=12, family="Arial"),
            coloraxis_showscale=False,
            margin=dict(l=300, r=150, t=100, b=50),
        )
        fig.show()

        df_table = (
            df_flow.groupby(["CIAP_Plot", "Specialty"])
            .size()
            .reset_index(name="Referral_Count")
            .sort_values(by=["Specialty", "Referral_Count"], ascending=[True, False])
            .reset_index(drop=True)
        )

        output_file = "Ophthalmology_Endocrinology_Flow_Table_EN.csv"
        df_table.to_csv(output_file, index=False, encoding="utf-8-sig")

        print(f"Success! Chart generated tracking {len(df_flow)} referrals.")
        print(f"Data table saved as: '{output_file}'.")

    else:
        print("No patient with a completed CIAP field was referred to Ophthalmology or Endocrinology in this dataset.")


# ---------------------------------------------------------------
# FIGURE 4 — CIAP flow to top 3 specialties
# ---------------------------------------------------------------
def figure_4_top3_specialties(df: pd.DataFrame) -> None:
    df_ev = df[df["Repeat Instrument"].notna()].copy()

    cols_ciap = [c for c in df.columns if "ciap" in c.lower()]
    if not cols_ciap:
        raise ValueError("No CIAP column was found.")
    col_ciap = cols_ciap[0]

    cols_espec = [c for c in df.columns if "Para qual especialidade" in c and "choice=" in c]

    df_ev = df_ev.dropna(subset=[col_ciap])
    df_ev = df_ev[df_ev[col_ciap].astype(str).str.strip() != ""]
    df_ev = df_ev[df_ev[col_ciap].astype(str).str.lower() != "nan"]

    total_records = []

    for _, row in df_ev.iterrows():
        current_ciap = str(row[col_ciap])

        for c in cols_espec:
            if row[c] == "Checked":
                specialty_name = c.split("choice=")[-1].replace(")", "").strip()
                total_records.append({"CIAP": current_ciap, "Specialty": specialty_name})

    df_total_flow = pd.DataFrame(total_records)

    if not df_total_flow.empty:
        top_3_specialties = df_total_flow["Specialty"].value_counts().nlargest(3).index.tolist()

        print("=" * 60)
        print("TOP 3 MOST DEMANDED SPECIALTIES:")
        for i, esp in enumerate(top_3_specialties, 1):
            print(f" {i}st place: {esp}" if i == 1 else f" {i}nd place: {esp}" if i == 2 else f" {i}rd place: {esp}")
        print("=" * 60)

        df_top3 = df_total_flow[df_total_flow["Specialty"].isin(top_3_specialties)].copy()

        color_map = {top_3_specialties[0]: 0, top_3_specialties[1]: 0.5, top_3_specialties[2]: 1}
        df_top3["Specialty_Color"] = df_top3["Specialty"].map(color_map)

        top_15_ciaps = df_top3["CIAP"].value_counts().nlargest(15).index
        df_top3["CIAP_Plot"] = np.where(df_top3["CIAP"].isin(top_15_ciaps), df_top3["CIAP"], "Other CIAPs (Grouped)")
        df_top3["CIAP_Plot"] = df_top3["CIAP_Plot"].apply(lambda x: x[:45] + "..." if len(x) > 45 else x)

        fig = px.parallel_categories(
            df_top3,
            dimensions=["CIAP_Plot", "Specialty"],
            color="Specialty_Color",
            color_continuous_scale=[(0.0, "#3498db"), (0.5, "#2ecc71"), (1.0, "#e74c3c")],
            labels={
                "CIAP_Plot": "Base Diagnosis (CIAP-2)",
                "Specialty": "Top 3 Specialties",
            },
            title=f"Referral Flow: CIAP -> Top 3 Specialties (N={len(df_top3)} flows)",
        )

        fig.update_layout(
            height=750,
            font=dict(size=12, family="Arial"),
            coloraxis_showscale=False,
            margin=dict(l=300, r=150, t=100, b=50),
        )
        fig.show()

        df_table = (
            df_top3.groupby(["CIAP_Plot", "Specialty"])
            .size()
            .reset_index(name="Referral_Count")
            .sort_values(by=["Specialty", "Referral_Count"], ascending=[True, False])
            .reset_index(drop=True)
        )

        output_file = "Top3_Specialties_Flow_Table_EN.csv"
        df_table.to_csv(output_file, index=False, encoding="utf-8-sig")

        print(f"\nSuccess! Chart generated tracking {len(df_top3)} referrals to the Top 3 specialties.")
        print(f"Data table saved as: '{output_file}'.\n")

    else:
        print("No valid referrals with CIAP filled in were found in the dataset.")


# ---------------------------------------------------------------
# FIGURE 5 — Superuser dispersion with AI-clustered outcomes
# ---------------------------------------------------------------
def figure_5_superusers_ai(df: pd.DataFrame) -> None:
    df_ev = df[df["Repeat Instrument"].notna()].copy()

    cols_ciap = [c for c in df.columns if "ciap" in c.lower()]
    col_ciap = cols_ciap[0] if cols_ciap else None

    cols_espec = [c for c in df.columns if "Para qual especialidade" in c and "choice=" in c]

    cols_outcome = [
        c for c in df.columns
        if any(word in c.lower() for word in ["desfecho", "conduta", "alta", "resolutividade", "plano"])
    ]
    col_outcome = cols_outcome[0] if cols_outcome else None

    if not col_outcome:
        raise ValueError("No Outcome or Conduct column was found for the AI analysis.")

    if cols_espec:
        referred_mask = df_ev[cols_espec].apply(lambda row: (row == "Checked").any(), axis=1)
        df_primary_care = df_ev[~referred_mask].copy()
    else:
        df_primary_care = df_ev.copy()

    df_primary_care = df_primary_care.dropna(subset=[col_outcome])
    df_primary_care = df_primary_care[df_primary_care[col_outcome].astype(str).str.strip() != ""]

    print("=" * 70)
    print(f"STARTING AI ANALYSIS OF OUTCOMES ({len(df_primary_care)} records)")
    print("=" * 70)

    outcome_texts = df_primary_care[col_outcome].astype(str).tolist()

    vectorizer = TfidfVectorizer(max_features=150, stop_words=None, ngram_range=(1, 2))
    X_vectors = vectorizer.fit_transform(outcome_texts)

    num_clusters = 4
    kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
    df_primary_care["AI_Cluster_ID"] = kmeans.fit_predict(X_vectors)

    ai_terms = vectorizer.get_feature_names_out()
    cluster_names = {}

    for i in range(num_clusters):
        centroid = kmeans.cluster_centers_[i]
        top_indices = centroid.argsort()[-3:][::-1]
        keywords = " + ".join([ai_terms[idx].title() for idx in top_indices])
        cluster_names[i] = f"[{keywords}]"

    df_primary_care["AI_Grouped_Outcome"] = df_primary_care["AI_Cluster_ID"].map(cluster_names)

    user_counts = df_ev["Record ID"].value_counts()
    top_10_ids = user_counts.head(10).index
    df_super = df_primary_care[df_primary_care["Record ID"].isin(top_10_ids)].copy()

    if col_ciap and not df_super.empty:
        df_super[col_ciap] = df_super[col_ciap].fillna("CIAP Not Reported")
        df_super["CIAP_Plot"] = df_super[col_ciap].astype(str).apply(lambda x: x[:40] + "..." if len(x) > 40 else x)
        df_super["Record_ID_Str"] = "Patient ID: " + df_super["Record ID"].astype(str)

        df_agg = (
            df_super.groupby(["Record_ID_Str", "CIAP_Plot", "AI_Grouped_Outcome"])
            .size()
            .reset_index(name="Consultation_Volume")
        )

        fig = px.scatter(
            df_agg,
            x="AI_Grouped_Outcome",
            y="CIAP_Plot",
            size="Consultation_Volume",
            color="Record_ID_Str",
            hover_name="Record_ID_Str",
            size_max=35,
            title="Superuser Dispersion: CIAP vs Conduct (AI Grouping)",
            labels={
                "AI_Grouped_Outcome": "Conduct / Outcome (AI)",
                "CIAP_Plot": "Diagnosis (CIAP-2)",
                "Record_ID_Str": "Superuser",
            },
        )

        fig.update_layout(
            height=600,
            font=dict(size=11, family="Arial"),
            margin=dict(l=280, r=50, t=80, b=120),
            xaxis_tickangle=-15,
            plot_bgcolor="rgba(240, 240, 240, 0.5)",
        )

        fig.update_xaxes(showgrid=True, gridwidth=1, gridcolor="white")
        fig.update_yaxes(showgrid=True, gridwidth=1, gridcolor="white")
        fig.show()

        df_table = df_agg.sort_values(
            by=["Consultation_Volume", "Record_ID_Str"], ascending=[False, True]
        ).reset_index(drop=True)

        output_file = "Superusers_AI_Dispersion_Table_EN.csv"
        df_table.to_csv(output_file, index=False, encoding="utf-8-sig")

        print("\nSuccess! Scatter plot generated for the Top 10 superusers.")
        print(f"Data table saved as: '{output_file}'.\n")

    else:
        print("\nWarning: there is not enough CIAP data for superusers, or the data is insufficient.")


# ---------------------------------------------------------------
# RUN EVERYTHING
# ---------------------------------------------------------------
def main():
    df = load_csv()
    figure_1_chronic_vs_injectables(df)
    figure_2_ciap_and_time(df)
    figure_3_ophthalmo_vs_endocrino(df)
    figure_4_top3_specialties(df)
    figure_5_superusers_ai(df)


main()